# Data Validation and Consistency Checks

## Objective

This notebook validates data quality and consistency across the raw, Bronze, Silver, and Gold datasets.

## Validation Scope

- Record count reconciliation
- Duplicate checks
- Null value checks
- Primary key validation
- Referential integrity checks
- Business rule validation
- Final Gold output verification

In [0]:
from pyspark.sql import functions as F


In [0]:
raw_base_path = "/Volumes/workspace/default/rideshare_data"
bronze_base_path = f"{raw_base_path}/bronze"
silver_path = f"{raw_base_path}/silver/ride_data"
gold_base_path = f"{raw_base_path}/gold"

raw_drivers_df = spark.read.option("header", True).csv(
    f"{raw_base_path}/drivers.csv"
)

raw_trips_df = spark.read.option("header", True).csv(
    f"{raw_base_path}/trips.csv"
)

raw_trip_logs_df = spark.read.option("header", True).csv(
    f"{raw_base_path}/trip_logs.csv"
)

bronze_drivers_df = spark.read.parquet(
    f"{bronze_base_path}/drivers"
)

bronze_trips_df = spark.read.parquet(
    f"{bronze_base_path}/trips"
)

bronze_trip_logs_df = spark.read.parquet(
    f"{bronze_base_path}/trip_logs"
)

silver_df = spark.read.parquet(silver_path)

print("All validation datasets loaded successfully.")

All validation datasets loaded successfully.


In [0]:
validation_counts = [
    ("Raw Drivers", raw_drivers_df.count()),
    ("Bronze Drivers", bronze_drivers_df.count()),
    ("Raw Trips", raw_trips_df.count()),
    ("Bronze Trips", bronze_trips_df.count()),
    ("Raw Trip Logs", raw_trip_logs_df.count()),
    ("Bronze Trip Logs", bronze_trip_logs_df.count()),
    ("Silver Records", silver_df.count())
]

validation_counts_df = spark.createDataFrame(
    validation_counts,
    ["dataset_stage", "record_count"]
)

display(validation_counts_df)

dataset_stage,record_count
Raw Drivers,150
Bronze Drivers,150
Raw Trips,150
Bronze Trips,150
Raw Trip Logs,150
Bronze Trip Logs,150
Silver Records,150


In [0]:
duplicate_checks = [
    (
        "Bronze Drivers Duplicate driver_id",
        bronze_drivers_df.count()
        - bronze_drivers_df.dropDuplicates(["driver_id"]).count()
    ),
    (
        "Bronze Trips Duplicate trip_id",
        bronze_trips_df.count()
        - bronze_trips_df.dropDuplicates(["trip_id"]).count()
    ),
    (
        "Bronze Trip Logs Duplicate log_id",
        bronze_trip_logs_df.count()
        - bronze_trip_logs_df.dropDuplicates(["log_id"]).count()
    ),
    (
        "Silver Duplicate trip_id",
        silver_df.count()
        - silver_df.dropDuplicates(["trip_id"]).count()
    )
]

duplicate_checks_df = spark.createDataFrame(
    duplicate_checks,
    ["validation_check", "duplicate_count"]
)

display(duplicate_checks_df)

validation_check,duplicate_count
Bronze Drivers Duplicate driver_id,0
Bronze Trips Duplicate trip_id,0
Bronze Trip Logs Duplicate log_id,0
Silver Duplicate trip_id,0


In [0]:
missing_driver_refs = (
    bronze_trips_df.alias("t")
    .join(
        bronze_drivers_df.alias("d"),
        F.col("t.driver_id") == F.col("d.driver_id"),
        "left_anti"
    )
    .count()
)

missing_trip_log_refs = (
    bronze_trips_df.alias("t")
    .join(
        bronze_trip_logs_df.alias("l"),
        F.col("t.trip_id") == F.col("l.trip_id"),
        "left_anti"
    )
    .count()
)

print("Trips with missing driver reference:", missing_driver_refs)
print("Trips with missing trip log reference:", missing_trip_log_refs)

Trips with missing driver reference: 0
Trips with missing trip log reference: 0


In [0]:
invalid_completed_timestamps = silver_df.filter(
    (F.col("trip_status") == "Completed")
    & (
        F.col("start_time").isNull()
        | F.col("end_time").isNull()
        | (F.col("end_time") < F.col("start_time"))
    )
).count()

invalid_cancelled_flags = silver_df.filter(
    (
        (F.col("trip_status") == "Cancelled")
        & (F.col("cancellation_flag") != 1)
    )
    |
    (
        (F.col("trip_status") == "Completed")
        & (F.col("cancellation_flag") != 0)
    )
).count()

invalid_numeric_values = silver_df.filter(
    (F.col("distance_km") < 0)
    | (F.col("fare_amount") < 0)
    | (F.col("delay_minutes") < 0)
    | (F.col("rating") < 0)
    | (F.col("rating") > 5)
).count()

print("Invalid completed-trip timestamps:", invalid_completed_timestamps)
print("Status and cancellation flag mismatches:", invalid_cancelled_flags)
print("Invalid numeric records:", invalid_numeric_values)

Invalid completed-trip timestamps: 0
Status and cancellation flag mismatches: 0
Invalid numeric records: 0


In [0]:
gold_validation = []

gold_table_names = [
    "driver_performance",
    "cancellation_analysis",
    "revenue_analysis",
    "delay_analysis",
    "high_demand_locations",
    "business_kpi"
]

for table_name in gold_table_names:
    table_path = f"{gold_base_path}/{table_name}"
    table_df = spark.read.parquet(table_path)

    gold_validation.append(
        (table_name, table_df.count())
    )

gold_validation_df = spark.createDataFrame(
    gold_validation,
    ["gold_table", "record_count"]
)

display(gold_validation_df)

gold_table,record_count
driver_performance,99
cancellation_analysis,99
revenue_analysis,5
delay_analysis,99
high_demand_locations,5
business_kpi,1


In [0]:
final_validation_results = [
    (
        "Raw to Bronze record count",
        "PASS"
        if (
            raw_drivers_df.count() == bronze_drivers_df.count()
            and raw_trips_df.count() == bronze_trips_df.count()
            and raw_trip_logs_df.count() == bronze_trip_logs_df.count()
        )
        else "FAIL"
    ),
    (
        "Duplicate primary keys",
        "PASS"
        if (
            bronze_drivers_df.count()
            - bronze_drivers_df.dropDuplicates(["driver_id"]).count()
            == 0
            and bronze_trips_df.count()
            - bronze_trips_df.dropDuplicates(["trip_id"]).count()
            == 0
            and bronze_trip_logs_df.count()
            - bronze_trip_logs_df.dropDuplicates(["log_id"]).count()
            == 0
            and silver_df.count()
            - silver_df.dropDuplicates(["trip_id"]).count()
            == 0
        )
        else "FAIL"
    ),
    (
        "Referential integrity",
        "PASS"
        if (
            missing_driver_refs == 0
            and missing_trip_log_refs == 0
        )
        else "FAIL"
    ),
    (
        "Completed trip timestamp validation",
        "PASS"
        if invalid_completed_timestamps == 0
        else "FAIL"
    ),
    (
        "Trip status and cancellation flag consistency",
        "PASS"
        if invalid_cancelled_flags == 0
        else "FAIL"
    ),
    (
        "Numeric value validation",
        "PASS"
        if invalid_numeric_values == 0
        else "FAIL"
    ),
    (
        "Silver record count validation",
        "PASS"
        if silver_df.count() == raw_trips_df.count()
        else "FAIL"
    ),
    (
        "Gold table availability",
        "PASS"
        if len(gold_validation) == 6
        else "FAIL"
    )
]

final_validation_df = spark.createDataFrame(
    final_validation_results,
    ["validation_check", "status"]
)

display(final_validation_df)

validation_check,status
Raw to Bronze record count,PASS
Duplicate primary keys,PASS
Referential integrity,PASS
Completed trip timestamp validation,PASS
Trip status and cancellation flag consistency,PASS
Numeric value validation,PASS
Silver record count validation,PASS
Gold table availability,PASS


## Data Validation Conclusion

All validation checks were completed successfully.

- Raw and Bronze record counts are consistent.
- No duplicate primary keys were found.
- Referential integrity checks passed.
- Completed trips contain valid timestamps.
- Trip status and cancellation flags are consistent.
- No invalid numeric values were detected.
- Silver record count matches the source trip count.
- All required Gold tables are available.

The data pipeline is consistent, reliable, and ready for CSV export, GitHub submission, and Streamlit deployment.

In [0]:
csv_output_path = "/Volumes/workspace/default/rideshare_data/final_csv"

spark.read.parquet(
    f"{gold_base_path}/driver_performance"
).coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    f"{csv_output_path}/driver_performance"
)

spark.read.parquet(
    f"{gold_base_path}/cancellation_analysis"
).coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    f"{csv_output_path}/cancellation_rate"
)

spark.read.parquet(
    f"{gold_base_path}/revenue_analysis"
).coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    f"{csv_output_path}/revenue_summary"
)

spark.read.parquet(
    f"{gold_base_path}/delay_analysis"
).coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    f"{csv_output_path}/delay_report"
)

spark.read.parquet(
    f"{gold_base_path}/high_demand_locations"
).coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    f"{csv_output_path}/high_demand_locations"
)

print("Final CSV outputs exported successfully.")

Final CSV outputs exported successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/,cancellation_rate/,0,1783751148243
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/,delay_report/,0,1783751148243
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/,driver_performance/,0,1783751148243
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/,high_demand_locations/,0,1783751148243
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/,revenue_summary/,0,1783751148243


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv/driver_performance"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/_SUCCESS,_SUCCESS,0,1783751100000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/_committed_7227914332966748667,_committed_7227914332966748667,113,1783751100000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/_started_7227914332966748667,_started_7227914332966748667,0,1783751100000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/part-00000-tid-7227914332966748667-f561987c-55bd-4eea-88f2-6a8962839b6d-299-1-c000.csv,part-00000-tid-7227914332966748667-f561987c-55bd-4eea-88f2-6a8962839b6d-299-1-c000.csv,6208,1783751100000


In [0]:
source_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "driver_performance/"
    "part-00000-tid-7227914332966748667-f561987c-55bd-4eea-88f2-6a8962839b6d-299-1-c000.csv"
)

destination_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "driver_performance.csv"
)

dbutils.fs.cp(source_file, destination_file)

print("driver_performance.csv created successfully.")

driver_performance.csv created successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/_SUCCESS,_SUCCESS,0,1783751102000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/_committed_9185246906388585818,_committed_9185246906388585818,113,1783751102000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/_started_9185246906388585818,_started_9185246906388585818,0,1783751102000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/part-00000-tid-9185246906388585818-994f2ded-aa5b-49bf-b6f8-ed01887a07a8-301-1-c000.csv,part-00000-tid-9185246906388585818-994f2ded-aa5b-49bf-b6f8-ed01887a07a8-301-1-c000.csv,2960,1783751102000


In [0]:
source_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "cancellation_rate/"
    "part-00000-tid-9185246906388585818-994f2ded-aa5b-49bf-b6f8-ed01887a07a8-301-1-c000.csv"
)

destination_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "cancellation_rate.csv"
)

dbutils.fs.cp(source_file, destination_file)

print("cancellation_rate.csv created successfully.")

cancellation_rate.csv created successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/_SUCCESS,_SUCCESS,0,1783751104000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/_committed_1542716305422883131,_committed_1542716305422883131,113,1783751103000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/_started_1542716305422883131,_started_1542716305422883131,0,1783751103000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/part-00000-tid-1542716305422883131-b5762940-0ef9-474d-b7a9-5568bf42259f-303-1-c000.csv,part-00000-tid-1542716305422883131-b5762940-0ef9-474d-b7a9-5568bf42259f-303-1-c000.csv,262,1783751103000


In [0]:
source_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "revenue_summary/"
    "part-00000-tid-1542716305422883131-b5762940-0ef9-474d-b7a9-5568bf42259f-303-1-c000.csv"
)

destination_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "revenue_summary.csv"
)

dbutils.fs.cp(source_file, destination_file)

print("revenue_summary.csv created successfully.")

revenue_summary.csv created successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv/delay_report"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/_SUCCESS,_SUCCESS,0,1783751105000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/_committed_5351894468131852883,_committed_5351894468131852883,113,1783751105000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/_started_5351894468131852883,_started_5351894468131852883,0,1783751105000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/part-00000-tid-5351894468131852883-15e9b612-cbd4-4d33-8d4d-1ec7c3fe1cb7-305-1-c000.csv,part-00000-tid-5351894468131852883-15e9b612-cbd4-4d33-8d4d-1ec7c3fe1cb7-305-1-c000.csv,3799,1783751105000


In [0]:
source_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "delay_report/"
    "part-00000-tid-5351894468131852883-15e9b612-cbd4-4d33-8d4d-1ec7c3fe1cb7-305-1-c000.csv"
)

destination_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "delay_report.csv"
)

dbutils.fs.cp(source_file, destination_file)

print("delay_report.csv created successfully.")

delay_report.csv created successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/_SUCCESS,_SUCCESS,0,1783751107000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/_committed_3393300242100220106,_committed_3393300242100220106,113,1783751107000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/_started_3393300242100220106,_started_3393300242100220106,0,1783751106000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/part-00000-tid-3393300242100220106-ea6aafc5-f333-4df2-b73a-cf573c16bcae-307-1-c000.csv,part-00000-tid-3393300242100220106-ea6aafc5-f333-4df2-b73a-cf573c16bcae-307-1-c000.csv,251,1783751107000


In [0]:
source_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "high_demand_locations/"
    "part-00000-tid-3393300242100220106-ea6aafc5-f333-4df2-b73a-cf573c16bcae-307-1-c000.csv"
)

destination_file = (
    "/Volumes/workspace/default/rideshare_data/final_csv/"
    "high_demand_locations.csv"
)

dbutils.fs.cp(source_file, destination_file)

print("high_demand_locations.csv created successfully.")

high_demand_locations.csv created successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/final_csv"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate/,cancellation_rate/,0,1783751729431
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/cancellation_rate.csv,cancellation_rate.csv,2960,1783751426000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report/,delay_report/,0,1783751729431
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/delay_report.csv,delay_report.csv,3799,1783751588000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance/,driver_performance/,0,1783751729431
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/driver_performance.csv,driver_performance.csv,6208,1783751271000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations/,high_demand_locations/,0,1783751729431
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/high_demand_locations.csv,high_demand_locations.csv,251,1783751680000
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary/,revenue_summary/,0,1783751729431
dbfs:/Volumes/workspace/default/rideshare_data/final_csv/revenue_summary.csv,revenue_summary.csv,262,1783751511000
